# Question 7

In [0]:
df = spark.read.csv(
    "/Volumes/cyntexa_dev/sales/my_volume/ecommerce_dirty.csv",
    header=True,
    inferSchema=True
)


In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *

df = (
    df
    # Remove completely duplicate rows
    .dropna(how='all')
    .dropDuplicates()

    # -------------------------
    # Customer Name
    # -------------------------
    .withColumn(
        "customer_name",
        when(
            col("customer_name").isNull(),
            lit("Unknown")
        ).otherwise(
            trim(col("customer_name"))
        )
    )

    # -------------------------
    # Order Date
    # Handle multiple date formats
    # -------------------------
    .withColumn(
        "order_date",
        coalesce(
    try_to_date(col("order_date"), "yyyy-MM-dd"),
    try_to_date(col("order_date"), "dd/MM/yyyy"),
    try_to_date(col("order_date"), "dd-MM-yyyy"),
    try_to_date(col("order_date"), "dd MMM yyyy"),
    try_to_date(col("order_date"), "MMMM dd, yyyy"),
    try_to_date(col("order_date"), "yyyy/MM/dd"),
    try_to_date(col("order_date"), "dd.MM.yyyy"),
    try_to_date(col("order_date"), "MMM-dd-yyyy"),
    try_to_date(col("order_date"), "dd MMMM yyyy")
)
    )

    # Remove rows where date could not be parsed
    .filter(col("order_date").isNotNull())
    # -------------------------
    # Quantity
    # Keep only valid positive quantities
    # -------------------------
    .withColumn(
        "quantity",
        col("quantity").try_cast("int")
    )

    .filter(col("quantity") > 0)

    

    # -------------------------
    # Price
    # Remove currency symbols / commas
    # Then cast to numeric
    # Invalid values become NULL
    # -------------------------
    .withColumn(
        "price",
        regexp_replace(
            col("price"),
            r"[$₹€,]",
            ""
        ).try_cast("double")
    )

    # Keep only valid positive prices
    .filter(col("price") > 0)

    # -------------------------
    # Category
    # -------------------------
    .withColumn(
        "category",
        when(
            col("category").isNull(),
            lit("unknown_category")
        ).otherwise(
            trim(lower(col("category")))
        )
    )

    # -------------------------
    # City
    # -------------------------
    .withColumn(
        "city",
        when(
            col("city").isNull(),
            lit("Unknown")
        ).otherwise(
            trim(lower(col("city")))
        )
    )

    # -------------------------
    # Payment Method
    # -------------------------
    .withColumn(
        "payment_method",
        when(
            col("payment_method").isNull(),
            lit("Unknown")
        ).otherwise(
            trim(lower(col("payment_method")))
        )
    )
)

df.display()


# Question 8


# Dev/Main Branching Strategy and Pull Request Review Workflow

## 1. Branching Strategy

The repository follows a two-level branching strategy:

* **`main`** – Stable and production-ready code.
* **`dev`** – Integration and testing branch.
* **Feature branches** – Created from `dev` for individual tasks.

Example:

```text
main
  │
  └── dev
       ├── feature/customer-kpi
       └── feature/orders-kpi
```

## 2. Development Workflow

1. Create the `dev` branch from `main`.
2. Give the required teammates access to the repository.
3. Each developer creates a separate feature branch from `dev`.
4. Developers work only on their assigned feature.
5. After completing the work, they commit and push their feature branch.
6. They create a Pull Request from their feature branch to `dev`.

Example:

```text
feature/customer-kpi → dev
feature/orders-kpi   → dev
```

## 3. Pull Request Review Workflow

Before merging a Pull Request into `dev`:

* Review the code changes.
* Compare the changes with the existing code.
* Verify that the KPI/business logic is correct.
* Check for unnecessary or duplicate code.
* Ensure the changes do not break existing functionality.
* Request changes if any issue is found.
* Approve the Pull Request when the changes are satisfactory.
* Merge the approved Pull Request into `dev`.

## 4. Merging into Main

After all required features are merged into `dev`:

1. Test the combined changes in `dev`.
2. Create a Pull Request from `dev` to `main`.
3. Review the changes again.
4. Ensure `dev` is stable and ready for release.
5. Approve and merge the Pull Request into `main`.

```text
Feature Branch
      │
      ▼
     dev
      │
      │  Testing + Review
      ▼
    main
```

This workflow keeps `main` stable while allowing multiple developers to work independently on different features and ensures that code is reviewed before it reaches the main branch.


# Question 9

##  KPIs -
- Total Revenue by Category
- Monthly Revenue + MoM Growth
- Average Order Value (AOV) Trend

In [0]:
from pyspark.sql.functions import *
df.groupBy('category').agg(sum(col('price')*col('quantity')).alias('total')).display()


In [0]:
from pyspark.sql.window import *
df_monthly = (
    df
    .withColumn("month", date_format("order_date", "yyyy-MM"))
    .groupBy("month")
    .agg(
        sum(col("price") * col("quantity")).alias("revenue")
    )
)

wind = Window.orderBy(col("revenue").desc())

df_monthly = (
    df_monthly
    .withColumn(
        "prev_revenue",
        lag("revenue").over(wind)
    )
    .withColumn(
        "growth_%",
        when(
            col("prev_revenue").isNull(),
            100
        ).otherwise(
            ((col("revenue") - col("prev_revenue")) / col("prev_revenue")) * 100
        )
    )
    .withColumn(
        "prev_revenue",
        coalesce(col("prev_revenue"), lit(0))
    )
)

df_monthly.display()


In [0]:
df_aov = (
    df
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .groupBy("year", "month")
    .agg(
        sum(col("price") * col("quantity")).alias("revenue"),
        countDistinct("order_id").alias("orders")
    )
    .withColumn(
        "aov",
       round( col("revenue") / col("orders"),2)
    )
)

display(df_aov)

### Data Quality Caveats:
The report is based on the cleaned dataset. Duplicate, invalid, and unparseable records were removed, while missing categorical values were replaced with "Unknown". Therefore, the reported metrics may differ from the original raw-data figures.